# BoardMind AI: Model Architecture & Training Optimization Experiments
**Notebook:** `05_whiteboard_model_experiments.ipynb`  
**Author:** Senior Computer Vision & ML Research Engineer  
**Hardware:** Intel Core i9-13900H CPU, 16GB RAM, Windows  

### Objectives:
1. Audit the existing baseline YOLOv8n checkpoint (`runs/whiteboard_detection_test/weights/best.pt`).
2. Evaluate model architectures suitable for CPU inference (YOLOv8n vs YOLO11n).
3. Investigate transfer learning and fine-tuning strategies.
4. Experiment with input image resolutions (320, 416, 512).
5. Optimize hyperparameters: optimizer (AdamW vs Auto/SGD), learning rate, and cosine scheduling.

In [ ]:
import sys
import os
import json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import torch
from ultralytics import YOLO

# Ensure project root and src/ are in sys.path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(PROJECT_ROOT / 'src'))

print(f"Project Root: {PROJECT_ROOT}")
print(f"PyTorch: {torch.__version__} | CUDA Available: {torch.cuda.is_available()}")

DATA_YAML = PROJECT_ROOT / 'datasets' / 'whiteboard' / 'data.yaml'
REGISTRY_FILE = PROJECT_ROOT / 'runs' / 'whiteboard_optimization' / 'experiment_registry.json'

## 1. Load Baseline Model & Validation Profile
We inspect the existing checkpoint trained in development stage 1.

In [ ]:
from evaluate_model import evaluate_checkpoint, benchmark_inference

baseline_ckpt = PROJECT_ROOT / 'runs' / 'whiteboard_detection_test' / 'weights' / 'best.pt'
print("Baseline checkpoint exists:", baseline_ckpt.exists())

baseline_model = YOLO(str(baseline_ckpt))
print("Model task:", baseline_model.task)
print("Class names:", baseline_model.names)

# Model parameters and FLOPs
baseline_model.info(detailed=False)

## 2. Model Architecture Comparison (YOLOv8n vs YOLO11n)
YOLO11 introduces C3k2 blocks and optimized SPPF with reduced parameter count and lower computational complexity (GFLOPs).

In [ ]:
# Compare architectures
m_v8 = YOLO('yolov8n.pt')
m_11 = YOLO('yolo11n.yaml')

print("--- YOLOv8n Profile ---")
m_v8.info()

print("\n--- YOLO11n Profile ---")
m_11.info()

## 3. Image Resolution Optimization (320 vs 416 vs 512)
We evaluate how input resolution impacts detection accuracy vs CPU latency (FPS).

In [ ]:
resolutions = [320, 416, 512]
val_images = list((PROJECT_ROOT / 'datasets' / 'whiteboard' / 'valid' / 'images').glob('*.*'))[:20]

benchmarks = {}
for res in resolutions:
    b = benchmark_inference(baseline_model, val_images, imgsz=res, iterations=20)
    benchmarks[res] = b
    print(f"Resolution {res}x{res}: {b['mean_latency_ms']} ms/image ({b['fps']} FPS)")

df_res = pd.DataFrame(benchmarks).T
df_res

## 4. Hyperparameter Optimization & Experiment Registry
Reviewing recorded experiments from `runs/whiteboard_optimization/experiment_registry.json`.

In [ ]:
if REGISTRY_FILE.exists():
    with open(REGISTRY_FILE, 'r') as f:
        registry = json.load(f)
    
    table_rows = []
    for name, rec in registry.items():
        m = rec['metrics_val']
        lat = rec.get('latency_benchmark', {})
        table_rows.append({
            'Experiment': name,
            'Model': rec['model_source'],
            'Image Size': rec['imgsz'],
            'Category': rec['category'],
            'Precision': m['precision'],
            'Recall': m['recall'],
            'F1': m.get('f1_score', 0),
            'mAP50': m['mAP50'],
            'mAP50-95': m['mAP50_95'],
            'Latency (ms)': lat.get('mean_latency_ms', m.get('total_latency_ms', 0)),
            'FPS': lat.get('fps', m.get('fps', 0))
        })
    
    df_exp = pd.DataFrame(table_rows)
    display(df_exp)
else:
    print("Registry file not found yet.")